# Income Risk and Wealth Inequality

The canonical Aiyagari model of lecture 4, solved with `GEModelTools`, here with an annual calibration.
Question: what does more idiosyncratic income risk do to the wealth distribution?

Log productivity follows an AR(1), $\log z_{t}=\rho_{z}\log z_{t-1}+\psi_{t}$ with $\psi_{t}\sim\mathcal{N}(0,\sigma_{\psi}^{2})$.
We set $\sigma_{\psi}=0.2\sqrt{1-\rho_{z}^{2}}$, so the standard deviation of $\log z$ is 0.2, and the experiment doubles it to 0.4.
Utility is CRRA with risk aversion $\sigma$.

1. Baseline ($\sigma=2$, $\rho_{z}=0.9$): compare the stationary equilibria in partial and general equilibrium, and compute the transition between them.
2. How does the answer depend on $\sigma$ and $\rho_{z}$?

The model is annual.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from HANCModel import HANCModelClass

## Settings

In [ ]:
plt.rcParams.update({'axes.grid':True,'grid.alpha':0.3,'font.size':12})
os.makedirs('figs',exist_ok=True)

std_log_z_high = 0.4

sigma_vec = [1.0,2.0,3.0,5.0]
rho_vec = [0.0,0.3,0.5,0.7,0.9]

## 1. Baseline


In [ ]:
model = HANCModelClass(name='baseline')
model.calibrate_ss()

print(f'beta = {model.par.beta:.4f}, delta = {model.par.delta*100:.2f} %, Gamma = {model.par.Gamma_ss:.4f}')
print(f'K = {model.ss.K:.2f}, Y = {model.ss.Y:.2f}, r = {model.ss.r*100:.2f} %')

In [ ]:
def wealth_distribution(D,a_grid):
    """Wealth shares, Gini coefficient and share at the borrowing constraint."""

    D_a = D.sum(axis=(0,1))
    cdf = np.cumsum(D_a)
    lorenz = np.cumsum(a_grid*D_a)/np.sum(a_grid*D_a)

    return {
        'bottom 50%':np.interp(0.5,cdf,lorenz),
        'middle 40%':np.interp(0.9,cdf,lorenz)-np.interp(0.5,cdf,lorenz),
        'top 10%':1-np.interp(0.9,cdf,lorenz),
        'top 1%':1-np.interp(0.99,cdf,lorenz),
        'top 0.1%':1-np.interp(0.999,cdf,lorenz),
        'Gini':1-2*np.trapezoid(np.insert(lorenz,0,0.0),np.insert(cdf,0,0.0)),
        'at constraint':D_a[0],
    }

pd.Series(wealth_distribution(model.ss.D,model.par.a_grid)).round(3)

### 1.1 More income risk: partial equilibrium

Copy the model and raise $\sigma_{\psi}$ so that the standard deviation of $\log z$ is `std_log_z_high`.
Keep $r$ and $w$ at their baseline values, and solve and simulate the household problem.
Print household wealth $A^{hh}$ and the wealth distribution.

In [ ]:


















# write your code here

**Solution:**

In [ ]:
model_pe = model.copy(name='PE')
model_pe.par.sigma_psi = std_log_z_high*np.sqrt(1-model.par.rho_z**2)
model_pe.solve_hh_ss()
model_pe.simulate_hh_ss()

print(f'A_hh = {model_pe.ss.A_hh:.2f}, r = {model_pe.ss.r*100:.2f} %')
pd.Series(wealth_distribution(model_pe.ss.D,model_pe.par.a_grid)).round(3)

### 1.2 More income risk: general equilibrium

Now let $r$ and $w$ adjust: find the new stationary equilibrium with `find_ss` (search for $K$ between 1 and 10), keeping $\beta$, $\Gamma$ and $\delta$ at their calibrated values.

In [ ]:






























# write your code here

**Solution:**

In [ ]:
model_ge = model.copy(name='GE')
model_ge.par.sigma_psi = std_log_z_high*np.sqrt(1-model.par.rho_z**2)
model_ge.find_ss(K_min=1.0,K_max=10.0)

print(f'K = {model_ge.ss.K:.2f}, r = {model_ge.ss.r*100:.2f} %')
pd.Series(wealth_distribution(model_ge.ss.D,model_ge.par.a_grid)).round(3)

### 1.3 Comparison

In [ ]:
def std_log_income(model):
    """Standard deviation of log productivity in the ergodic distribution."""

    D_z = model.ss.D.sum(axis=(0,2))
    log_z = np.log(model.par.z_grid)
    return np.sqrt(np.sum(D_z*(log_z-np.sum(D_z*log_z))**2))

models = [model,model_pe,model_ge]

table = pd.DataFrame({
    m.name:{'sigma_psi':m.par.sigma_psi,'std. log income':std_log_income(m),'A_hh':m.ss.A_hh,'r (%)':m.ss.r*100,
            **wealth_distribution(m.ss.D,m.par.a_grid)}
    for m in models})

table.round(3)

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4.5))

for m in models:

    D_a = m.ss.D.sum(axis=(0,1))
    cdf = np.insert(np.cumsum(D_a),0,0.0)
    lorenz = np.insert(np.cumsum(m.par.a_grid*D_a)/np.sum(m.par.a_grid*D_a),0,0.0)

    axes[0].plot(cdf,lorenz,label=f'{m.name}, std. log z = {std_log_income(m):.1f}')
    axes[1].plot(np.insert(m.par.a_grid,0,0.0),cdf,label=m.name)

axes[0].plot([0,1],[0,1],color='black',ls=':',label='perfect equality')
axes[0].set_xlabel('share of households (poorest first)')
axes[0].set_ylabel('share of wealth')
axes[0].set_title('Lorenz curve')
axes[0].legend(frameon=False)

axes[1].set_xscale('symlog')
axes[1].set_xlabel('assets, $a_t$')
axes[1].set_ylabel('CDF')
axes[1].set_title('Wealth distribution')

fig.tight_layout()
fig.savefig('figs/income_risk_wealth.pdf')

### 1.4 Transition

At $t=0$ income risk rises unexpectedly and permanently, from the baseline steady state.
Compute the transition path to the GE steady state of section 1.2:
- the terminal steady state is `model_ge`, so compute its Jacobians
- the initial state is the baseline: pass `ini={'Dbeg':...,'K':...}` with the baseline values to `find_transition_path`, with `shocks=[]`

The Rouwenhorst grid keeps the same $N_z$ states with the same ergodic weights, so a household in state $i_z$ stays in state $i_z$ and its productivity jumps to the new, more dispersed grid.

In [ ]:
# write your code here

**Solution:**

In [ ]:
model_ge.compute_jacs()
model_ge.find_transition_path(shocks=[],ini={'Dbeg':model.ss.Dbeg.copy(),'K':model.ss.K},do_print=True)

In [ ]:
path = model_ge.path
years = np.arange(model_ge.par.T)
shares_path = pd.DataFrame([wealth_distribution(path.D[t],model_ge.par.a_grid) for t in range(model_ge.par.T)])

fig, axes = plt.subplots(2,3,figsize=(15,8))

for ax,varname,title in zip(axes[0],['K','r','w'],['capital, $K_t$','interest rate, %','wage, $w_t$']):
    fac = 100 if varname == 'r' else 1
    ax.plot(years,path.__dict__[varname][:,0]*fac)
    ax.axhline(model.ss.__dict__[varname]*fac,color='black',ls=':',label='baseline')
    ax.axhline(model_ge.ss.__dict__[varname]*fac,color='black',ls='--',label='new steady state')
    ax.set_title(title)

for ax,share in zip(axes[1],['top 10%','top 1%','top 0.1%']):
    ax.plot(years,shares_path[share]*100)
    ax.axhline(table.loc[share,'baseline']*100,color='black',ls=':')
    ax.axhline(table.loc[share,'GE']*100,color='black',ls='--')
    ax.set_title(f'{share} wealth share, %')

axes[0,0].legend(frameon=False)
for ax in axes.ravel():
    ax.set_xlabel('years')
    ax.set_xlim(0,100)

fig.tight_layout()
fig.savefig('figs/income_risk_transition.pdf')

## 2. The role of risk aversion and persistence

For every $\sigma$ in `sigma_vec` and every $\rho_{z}$ in `rho_vec`:
1. set $\sigma$, $\rho_{z}$ and $\sigma_{\psi}=0.2\sqrt{1-\rho_{z}^{2}}$, and recalibrate $\beta$ with `calibrate_ss`
2. double the standard deviation of $\log z$ to 0.4 and find the new stationary equilibrium with `find_ss` (search for $K$ between 3 and 6), keeping $\beta$, $\Gamma$ and $\delta$ fixed
3. store the change in the top 1% share (in percentage points) and in the Gini coefficient

Fill two tables with $\sigma$ in rows and $\rho_{z}$ in columns. The loop takes about 2 minutes.

In [ ]:
d_top1 = pd.DataFrame(index=pd.Index(sigma_vec,name='sigma'),columns=pd.Index(rho_vec,name='rho_z'),dtype=float)
d_gini = pd.DataFrame(index=pd.Index(sigma_vec,name='sigma'),columns=pd.Index(rho_vec,name='rho_z'),dtype=float)

for sigma in sigma_vec:
    for rho_z in rho_vec:

        # a. baseline
        model_ = model.copy()


























# write your code here

**Solution:**

In [ ]:
d_top1 = pd.DataFrame(index=pd.Index(sigma_vec,name='sigma'),columns=pd.Index(rho_vec,name='rho_z'),dtype=float)
d_gini = pd.DataFrame(index=pd.Index(sigma_vec,name='sigma'),columns=pd.Index(rho_vec,name='rho_z'),dtype=float)

for sigma in sigma_vec:
    for rho_z in rho_vec:

        # a. baseline
        model_ = model.copy()
        model_.par.sigma = sigma
        model_.par.rho_z = rho_z
        model_.par.sigma_psi = 0.2*np.sqrt(1-rho_z**2)
        model_.calibrate_ss()
        baseline = wealth_distribution(model_.ss.D,model_.par.a_grid)

        # b. more income risk
        model_.par.sigma_psi = std_log_z_high*np.sqrt(1-rho_z**2)
        model_.find_ss(K_min=3.0,K_max=6.0)
        high_risk = wealth_distribution(model_.ss.D,model_.par.a_grid)

        d_top1.loc[sigma,rho_z] = (high_risk['top 1%']-baseline['top 1%'])*100
        d_gini.loc[sigma,rho_z] = high_risk['Gini']-baseline['Gini']

        print(f'sigma = {sigma}, rho_z = {rho_z}: beta = {model_.par.beta:.4f}, K = {model_.ss.K:.2f}, r = {model_.ss.r*100:.2f} %')

In [ ]:
print('change in the top 1% wealth share, percentage points')
d_top1.round(2)

In [ ]:
print('change in the Gini coefficient of wealth')
d_gini.round(3)